<a href="https://colab.research.google.com/github/Chellanadimuthu/ipl-transformer-chatbot/blob/main/ipl_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("GPU:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU: False
Device: CPU


In [4]:
from google.colab import files

uploaded = files.upload()

Saving ipl_dataset.txt to ipl_dataset.txt


In [6]:
import re

# Read the content of the uploaded file into the 'text' variable
with open("ipl_dataset.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

tokens = re.findall(r"\w+|[^\w\s]", text)

vocab = sorted(set(tokens))

word_to_idx = {w:i for i,w in enumerate(vocab)}
idx_to_word = {i:w for w,i in word_to_idx.items()}

encoded = [word_to_idx[t] for t in tokens]

vocab_size = len(vocab)

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 1850


In [7]:
import torch

SEQ_LEN = 20

X = []
Y = []

for i in range(len(encoded) - SEQ_LEN):

    X.append(encoded[i:i+SEQ_LEN])

    Y.append(encoded[i+1:i+SEQ_LEN+1])

X = torch.tensor(X)
Y = torch.tensor(Y)

print(X.shape)
print(Y.shape)

torch.Size([11101, 20])
torch.Size([11101, 20])


In [8]:
import torch.nn as nn

class MiniGPT(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 128)

        self.pos_embedding = nn.Parameter(
            torch.zeros(1, 512, 128)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=4,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=4
        )

        self.fc = nn.Linear(
            128,
            vocab_size
        )

    def forward(self, x):

        seq_len = x.shape[1]

        x = self.embedding(x)

        x = x + self.pos_embedding[:, :seq_len]

        x = self.transformer(x)

        return self.fc(x)

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MiniGPT(vocab_size).to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

print(model)

MiniGPT(
  (embedding): Embedding(1850, 128)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=128, out_features=1850, bias=True)
)


In [10]:
EPOCHS = 10
BATCH_SIZE = 32

for epoch in range(EPOCHS):

    total_loss = 0

    model.train()

    for i in range(0, len(X), BATCH_SIZE):

        xb = X[i:i+BATCH_SIZE].to(device)
        yb = Y[i:i+BATCH_SIZE].to(device)

        logits = model(xb)

        loss = loss_fn(
            logits.reshape(-1, vocab_size),
            yb.reshape(-1)
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS}  Loss={total_loss:.4f}"
    )

Epoch 1/10  Loss=2110.1969
Epoch 2/10  Loss=1720.7085
Epoch 3/10  Loss=1503.3244
Epoch 4/10  Loss=1318.3312
Epoch 5/10  Loss=1155.7312
Epoch 6/10  Loss=1008.7386
Epoch 7/10  Loss=872.1109
Epoch 8/10  Loss=744.7817
Epoch 9/10  Loss=636.5599
Epoch 10/10  Loss=541.8282


In [11]:
torch.save(
    {
        "model_state": model.state_dict(),
        "word_to_idx": word_to_idx,
        "idx_to_word": idx_to_word
    },
    "ipl_llm.pth"
)

print("Model Saved")

Model Saved


In [12]:
checkpoint = torch.load(
    "ipl_llm.pth",
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state"]
)

model.eval()

MiniGPT(
  (embedding): Embedding(1850, 128)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=128, out_features=1850, bias=True)
)

In [13]:
def generate(seed, max_tokens=50):

    words = seed.lower().split()

    ids = [
        word_to_idx[w]
        for w in words
        if w in word_to_idx
    ]

    if len(ids) == 0:
        return "Unknown words"

    x = torch.tensor(
        [ids[-SEQ_LEN:]],
        device=device
    )

    with torch.no_grad():

        for _ in range(max_tokens):

            logits = model(x)

            next_id = logits[
                0,
                -1
            ].argmax().item()

            ids.append(next_id)

            x = torch.tensor(
                [ids[-SEQ_LEN:]],
                device=device
            )

    return " ".join(
        idx_to_word[i]
        for i in ids
    )

In [ ]:
while True:

    q = input("You: ")

    if q.lower() == "exit":
        break

    answer = generate(q, 30)

    print("\nBot:", answer)
    print()


Bot: who is dhoni is five is used due is used due is warriors is used due is warriors is warriors is warriors is warriors is warriors is used involved is used involved is


Bot: raina background background notice legendary moments that define modern new format competition attracting world - class spinners who follow every single year due to pick up premier league can come every


Bot: rohit career cricket and career cricket and career cricket and career cricket and career cricket and career cricket and cricket and cricket and cricket and cricket and cricket and cricket and


Bot: 2010 players players players players players players players players players players players players players players players players players players players players players players players players players players players players players players


Bot: ipl 2010 ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl ipl



In [ ]:
from google.colab import files

files.download("ipl_llm.ipynb")